In [1]:
from pyspark.sql import SparkSession
import pyspark

# Define the GCS Connector version (Hadoop 3 compatible)
GCS_CONNECTOR_VERSION = "hadoop3-2.2.22" # Check Maven for the absolute latest

spark = SparkSession.builder \
    .appName("GCP-Connect") \
    .config("spark.jars.packages", f"com.google.cloud.bigdataoss:gcs-connector:{GCS_CONNECTOR_VERSION}") \
    .config("spark.hadoop.fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem") \
    .config("spark.hadoop.google.cloud.auth.service.account.enable", "true") \
    .config("spark.hadoop.google.cloud.auth.service.account.json.keyfile", "/home/mounir_salam/python_projects/DE_Zoomcamp/pipeline/keys/de-zoomcamp-key.json") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/27 21:42:58 WARN Utils: Your hostname, MounirLenovo2, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/02/27 21:42:58 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/mounir_salam/python_projects/DE_Zoomcamp/.venv/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/mounir_salam/.ivy2.5.2/cache
The jars for the packages stored in: /home/mounir_salam/.ivy2.5.2/jars
com.google.cloud.bigdataoss#gcs-connector added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-98a710dc-f38c-4b2c-8fae-024ff81ec4cf;1.0
	confs: [default]
	found com.google.cloud.bigdataoss#gcs-connector;hadoop3-2.2.22 in central
	found com.google.api-client#google-api-client-jackson2;2.0.1 in central
	found com.google.a

In [19]:
df_green = spark.read.parquet("gs://de-zoomcamp-484617-function_bucket/green_tripdata_2020-{01,02,03}.parquet")
df_yellow = spark.read.parquet("gs://de-zoomcamp-484617-function_bucket/yellow_tripdata_2020_{04,05,06}.parquet")

df_green.cache()
df_yellow.cache()

26/02/27 23:18:12 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: gs://de-zoomcamp-484617-function_bucket/green_tripdata_2020-{01,02,03}.parquet.
java.io.FileNotFoundException: File not found: gs://de-zoomcamp-484617-function_bucket/green_tripdata_2020-{01,02,03}.parquet
	at com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystemBase.lambda$getFileStatus$9(GoogleHadoopFileSystemBase.java:1077)
	at com.google.cloud.hadoop.fs.gcs.GhfsStorageStatistics.trackDuration(GhfsStorageStatistics.java:102)
	at com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystemBase.getFileStatus(GoogleHadoopFileSystemBase.java:1062)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1Batc

DataFrame[VendorID: bigint, tpep_pickup_datetime: timestamp_ntz, tpep_dropoff_datetime: timestamp_ntz, passenger_count: double, trip_distance: double, RatecodeID: double, store_and_fwd_flag: string, PULocationID: bigint, DOLocationID: bigint, payment_type: bigint, fare_amount: double, extra: double, mta_tax: double, tip_amount: double, tolls_amount: double, improvement_surcharge: double, total_amount: double, congestion_surcharge: double, airport_fee: void]

In [20]:
df_green.show()
df_yellow.show()

+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+
|VendorID|lpep_pickup_datetime|lpep_dropoff_datetime|store_and_fwd_flag|RatecodeID|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|extra|mta_tax|tip_amount|tolls_amount|ehail_fee|improvement_surcharge|total_amount|payment_type|trip_type|congestion_surcharge|
+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+
|       2| 2019-12-18 15:52:30|  2019-12-18 15:54:39|                 N|       1.0|         264|         264|            5.0|          0.0|        3.5|  0.5|    0.

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2020-04-01 00:41:22|  2020-04-01 01:01:53|            1.0|          1.2|       1.0|                 N|          41|          24|           2|        5.5|  0.5|    0.5|       0.

In [ ]:
# find records with pickup datetime include 2020-01-01
from pyspark.sql.functions import col, to_date
from pyspark.sql.functions import spark_partition_id
    
# df_green.explain(mode="formatted")
# df_green.rdd.getNumPartitions()

df_green.filter(to_date(col("lpep_pickup_datetime")) == "2020-01-01").show()

# df_green.withColumn("partition_num", spark_partition_id()) \
#         .groupBy("partition_num") \
#         .count() \
#         .show()

+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+
|VendorID|lpep_pickup_datetime|lpep_dropoff_datetime|store_and_fwd_flag|RatecodeID|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|extra|mta_tax|tip_amount|tolls_amount|ehail_fee|improvement_surcharge|total_amount|payment_type|trip_type|congestion_surcharge|
+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+
|       2| 2020-01-01 00:45:58|  2020-01-01 00:56:39|                 N|       5.0|          66|          65|            2.0|         1.28|       20.0|  0.0|    0.

In [28]:
df_yellow.filter(to_date(col("tpep_pickup_datetime")) == "2020-04-01").show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2020-04-01 00:41:22|  2020-04-01 01:01:53|            1.0|          1.2|       1.0|                 N|          41|          24|           2|        5.5|  0.5|    0.5|       0.

In [30]:
df_green.filter(col("PULocationID") == 41).show()

+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+
|VendorID|lpep_pickup_datetime|lpep_dropoff_datetime|store_and_fwd_flag|RatecodeID|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|extra|mta_tax|tip_amount|tolls_amount|ehail_fee|improvement_surcharge|total_amount|payment_type|trip_type|congestion_surcharge|
+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+
|       2| 2020-01-01 00:52:18|  2020-01-01 01:09:58|                 N|       1.0|          41|         127|            1.0|         5.67|       19.0|  0.5|    0.

In [31]:
df_green = df_green \
    .withColumnRenamed("lpep_pickup_datetime", "pickup_datetime") \
    .withColumnRenamed("lpep_dropoff_datetime", "dropoff_datetime")

df_green.columns

['VendorID',
 'pickup_datetime',
 'dropoff_datetime',
 'store_and_fwd_flag',
 'RatecodeID',
 'PULocationID',
 'DOLocationID',
 'passenger_count',
 'trip_distance',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'ehail_fee',
 'improvement_surcharge',
 'total_amount',
 'payment_type',
 'trip_type',
 'congestion_surcharge']

In [32]:
df_yellow = df_yellow \
    .withColumnRenamed("tpep_pickup_datetime", "pickup_datetime") \
    .withColumnRenamed("tpep_dropoff_datetime", "dropoff_datetime")

df_yellow.columns

['VendorID',
 'pickup_datetime',
 'dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'airport_fee']

In [33]:
common_columns = []

yello_cols = set(df_yellow.columns)

for col in df_green.columns:
    if col in yello_cols:
        common_columns.append(col)

print(common_columns)

['VendorID', 'pickup_datetime', 'dropoff_datetime', 'store_and_fwd_flag', 'RatecodeID', 'PULocationID', 'DOLocationID', 'passenger_count', 'trip_distance', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'payment_type', 'congestion_surcharge']


In [34]:
from pyspark.sql import functions as F

df_green_sel = df_green \
    .select(common_columns) \
    .withColumn("service_type", F.lit('green'))

In [35]:
df_yellow_sel = df_yellow \
    .select(common_columns) \
    .withColumn("service_type", F.lit('yellow'))

In [36]:
df_trips_data = df_green_sel.unionByName(df_yellow_sel)

In [38]:
df_trips_data.groupBy("service_type").count().show()

+------------+-------+
|service_type|  count|
+------------+-------+
|       green|1069898|
|      yellow|1136285|
+------------+-------+



In [41]:
df_trips_data.createOrReplaceTempView("trips_data")

In [50]:
df_result = spark.sql("""
    SELECT
          -- Revenue Grouping
          PULocationID AS revenue_zone,
          date_trunc('month', pickup_datetime) AS revenue_month,
          service_type,

          -- Revenue Calculation
          SUM(fare_amount) AS revenue_monthly_fare,
          SUM(extra) AS revenue_monthly_extra,
          SUM(mta_tax) AS revenue_monthly_mta_tax,
          SUM(tip_amount) AS revenue_monthly_tip,
          SUM(tolls_amount) AS revenue_monthly_tolls,
          SUM(improvement_surcharge) AS revenue_monthly_improvement_surcharge,
          SUM(total_amount) AS revenue_monthly_total_amount,
          SUM(congestion_surcharge) AS revenue_monthly_congestion_surcharge,
          
          -- Additional Metrics
          AVG(passenger_count) AS avg_monthly_passenger_count,
          AVG(trip_distance) AS avg_monthly_trip_distance
            
    FROM trips_data
    GROUP BY 1, 2, 3
""")

In [51]:
df_result.show()

+------------+-------------------+------------+--------------------+---------------------+-----------------------+-------------------+---------------------+-------------------------------------+----------------------------+------------------------------------+---------------------------+-------------------------+
|revenue_zone|      revenue_month|service_type|revenue_monthly_fare|revenue_monthly_extra|revenue_monthly_mta_tax|revenue_monthly_tip|revenue_monthly_tolls|revenue_monthly_improvement_surcharge|revenue_monthly_total_amount|revenue_monthly_congestion_surcharge|avg_monthly_passenger_count|avg_monthly_trip_distance|
+------------+-------------------+------------+--------------------+---------------------+-----------------------+-------------------+---------------------+-------------------------------------+----------------------------+------------------------------------+---------------------------+-------------------------+
|          72|2020-01-01 00:00:00|       green|  37109.

In [ ]:
df_result.write.parquet('data/reports/revenue/')

# OR Control number of partitions created from the write operation
# df_result.coalesce(1).write.parquet('data/reports/revenue/') # This will create a single output file, but be cautious with large datasets as it can lead to performance issues.